# Phase 6c Wave 3: When Things Fall, Do They All Fall the Same?

**A non-specialist tour of the equivalence principle and what it can — and cannot — tell us about dark matter.**

Companion notebook to `papers/paper34_equivalence_principle/paper_draft.tex`.

## The thought experiment

Galileo (apocryphally) dropped two balls from the Tower of Pisa and noticed they hit the ground together. Einstein, two and a half centuries later, took this seriously: he called the universality of free fall his "happiest thought" and built general relativity around it. The principle has three increasingly strong forms:

- **Weak Equivalence Principle (WEP):** all test masses fall identically.
- **Einstein Equivalence Principle (EEP):** WEP, plus all non-gravitational physics is the same in any local free-fall frame.
- **Strong Equivalence Principle (SEP):** EEP, plus the same statement applied to bodies whose own gravitational binding is significant.

Modern satellite experiments have tested WEP to extraordinary precision. The MICROSCOPE mission (2017) put the bound at $\eta < 10^{-15}$, where $\eta$ is the fractional difference in free-fall acceleration between two test masses. STEP-class follow-ups aim for $\eta \sim 10^{-18}$.

## Why this paper exists

The project's emergent-gravity framework has identified six different dark-sector / hidden-sector mechanisms — and until now, no one had said clearly which of them would *predict* an EP violation. This notebook walks through the classification: out of six mechanisms, exactly two violate the equivalence principle, and both are *vestigial-phase* phenomena (residues of an emergent-gravity broken phase). The other four are EP-respecting.

**The payload:** a future satellite EP test landing in the MICROSCOPE-to-STEP window would *uniquely* signal vestigial physics. EP precision becomes a direct discriminator.

## 1. The setup

Six mechanisms have been studied in earlier waves of this project. Two of them carry residual coupling differences between bosons and fermions — vestigial signatures of a phase where the metric and tetrad fields had different vacuum expectation values. The other four don't. Let's look at the experimental anchors first.

In [ ]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.equivalence_principle import (
    EPLevel, EPMechanism,
    MICROSCOPE_BOUND, STEP_TARGET,
    VESTIGIAL_PHASE_ETA_MAX, VESTIGIAL_RELICS_ETA,
    violation_level, violates_at,
    EP_VIOLATION_MATRIX,
)
from src.core.visualizations import COLORS, fig_ep_violation_matrix

print('Equivalence-principle test scales (Eötvös ratio η):')
print(f'  η = 1.0           — full vestigial phase (already excluded by MICROSCOPE)')
print(f'  η ~ 10⁻¹⁵         — MICROSCOPE bound  (Touboul 2017, current best)')
print(f'  η ~ 10⁻¹⁸         — STEP-class projected design sensitivity (white-paper target)')
print(f'  η ~ 10⁻¹⁸         — vestigial-relics residual prediction (project)')
print()
print(f'  → vestigial relics sit at the projected STEP-class design sensitivity, well below MICROSCOPE.')

## 2. The six mechanisms in plain language

**Vestigial differential coupling (violates WEP, $\eta = 1$).** In a phase where the metric and the tetrad don't quite agree, bosons and fermions "see" different effective gravity. The maximum signature is a maximal Eötvös ratio. *The η ≈ 1 maximal endpoint of this mechanism is already ruled out by MICROSCOPE; suppressed regimes of the same mechanism are the relics described next.*

**Vestigial relics (violates WEP, $\eta \sim 10^{-18}$).** The same physics, but suppressed: only topological-defect remnants survive a transition into the full-tetrad phase. Predicted residual: $\eta \sim 10^{-18}$ — *at the projected design sensitivity discussed for STEP-class satellite missions*.

**FangGu torsion-trace coupling (satisfies all EP).** A torsion-mediated dark-matter mechanism. Its failure mode is kinematic ($w_{FG} = 1/3$ doesn't match observed CDM $w \approx 0$), not EP-related.

**Fracton subdiffusion (satisfies all EP).** Fracton phases give universal subdiffusive transport — uniform across all matter, no preferred-direction or species-dependent coupling.

**SFDM Thomas–Fermi (satisfies all EP).** Berezhiani–Khoury superfluid dark matter as a single coherent scalar field — uniform coupling.

**Hidden-sector ℤ₁₆ singlet (satisfies all EP).** A standard-model-singlet sector decoupled at low energies — gravity sees only its energy-momentum, not flavor-dependent couplings.

In [2]:
labels = {
    EPMechanism.vestigialDifferentialCoupling: 'Vestigial differential coupling   (η=1, max)',
    EPMechanism.vestigialReliscSTEPClass:      'Vestigial relics                  (η~10⁻¹⁸)',
    EPMechanism.fangGuTorsionTrace:            'FangGu torsion-trace              (kinematic failure mode)',
    EPMechanism.fractonSubdiffusion:           'Fracton subdiffusion              (universal mobility)',
    EPMechanism.sfdmThomasFermi:               'SFDM Thomas-Fermi                 (single scalar)',
    EPMechanism.hiddenSectorZ16Singlet:        'Hidden-sector ℤ₁₆ singlet         (SM-decoupled)',
}
for m in EPMechanism:
    vl = violation_level(m)
    status = 'VIOLATES WEP+EEP+SEP' if vl is not None else 'satisfies all EP'
    print(f'  {labels[m]:55s}  →  {status}')

  Vestigial differential coupling   (η=1, max)             →  VIOLATES WEP+EEP+SEP
  Vestigial relics                  (η~10⁻¹⁸)              →  VIOLATES WEP+EEP+SEP
  FangGu torsion-trace              (kinematic failure mode)  →  satisfies all EP
  Fracton subdiffusion              (universal mobility)   →  satisfies all EP
  SFDM Thomas-Fermi                 (single scalar)        →  satisfies all EP
  Hidden-sector ℤ₁₆ singlet         (SM-decoupled)         →  satisfies all EP


## 3. The 6×3 matrix

The structural fact: among the six, *exactly two* violate WEP, and they are precisely the two vestigial mechanisms. There are no "WEP-only" or "SEP-only" violators — once you violate WEP, you violate all three by the implication chain. This is encoded as the Lean theorem `violatesAt_mono`.

We call the structural conclusion `ep_violation_is_vestigial_only`.

In [ ]:
header = f'{"mechanism":40s} | {"WEP":^11} | {"EEP":^11} | {"SEP":^11}'
print(header)
print('-' * len(header))
for m in EPMechanism:
    short_name = m.name.replace('vestigialDifferentialCoupling', 'vestigial differential coupling')\
                       .replace('vestigialReliscSTEPClass', 'vestigial relics (STEP-class)')\
                       .replace('fangGuTorsionTrace', 'FangGu torsion-trace')\
                       .replace('fractonSubdiffusion', 'fracton subdiffusion')\
                       .replace('sfdmThomasFermi', 'SFDM Thomas-Fermi')\
                       .replace('hiddenSectorZ16Singlet', 'hidden-sector ℤ₁₆ singlet')
    row = f'{short_name:40s}'
    for L in EPLevel:
        cell = '✗ violates' if EP_VIOLATION_MATRIX[m][L] else '✓ satisfies'
        row += f' | {cell:^11}'
    print(row)

print()
print('Pattern: only the two vestigial rows show "✗ violates". Within this substrate,')
print('a future EP detection in the MICROSCOPE-to-STEP precision window would point to')
print('vestigial physics — at the projected STEP-class design sensitivity, with the')
print('honest caveat that the classification is substrate-specific.')

## 4. The numerical content

The structural classification only matters if the numbers work out. Two quantitative statements are formalized in Lean (both proved by `norm_num`):

- The full vestigial phase predicts $\eta = 1$, *fifteen orders above MICROSCOPE*. Already excluded.
- The vestigial-relics phase predicts $\eta \sim 10^{-18}$, *three orders below MICROSCOPE* (so MICROSCOPE wouldn't see it) but at the *projected STEP-class design sensitivity*.

In [4]:
import math

print('Vestigial phase η = 1 vs MICROSCOPE bound:')
print(f'  predicted η_max = {VESTIGIAL_PHASE_ETA_MAX}')
print(f'  MICROSCOPE      = {MICROSCOPE_BOUND:.0e}')
ratio_excluded = math.log10(VESTIGIAL_PHASE_ETA_MAX / MICROSCOPE_BOUND)
print(f'  → predicted is {ratio_excluded:.0f} orders above bound — already EXCLUDED.')

print()
print('Vestigial relics η ~ 10⁻¹⁸ vs MICROSCOPE / STEP:')
print(f'  predicted η     = {VESTIGIAL_RELICS_ETA:.0e}')
print(f'  MICROSCOPE      = {MICROSCOPE_BOUND:.0e}')
print(f'  STEP target     = {STEP_TARGET:.0e}')
ratio_below = -math.log10(MICROSCOPE_BOUND / VESTIGIAL_RELICS_ETA)
print(f'  → predicted is {-ratio_below:.0f} orders BELOW MICROSCOPE  (so MICROSCOPE cannot see it)')
print(f'  → predicted is AT the STEP-class design sensitivity        (so STEP would see it).')

Vestigial phase η = 1 vs MICROSCOPE bound:
  predicted η_max = 1.0
  MICROSCOPE      = 1e-15
  → predicted is 15 orders above bound — already EXCLUDED.

Vestigial relics η ~ 10⁻¹⁸ vs MICROSCOPE / STEP:
  predicted η     = 1e-18
  MICROSCOPE      = 1e-15
  STEP target     = 1e-18
  → predicted is 3 orders BELOW MICROSCOPE  (so MICROSCOPE cannot see it)
  → predicted is AT the STEP-class design sensitivity        (so STEP would see it).


## 5. Visualizing the matrix and the η scales

**Left panel.** The 6×3 violation matrix as a heatmap. The amber band on top — two rows of "violates" — are the vestigial mechanisms. The four-row gray band underneath is the EP-respecting set.

**Right panel.** Four log-scale bars showing the η values that matter. The dotted blue line at $\log_{10}(\eta) = -15$ is the MICROSCOPE bound. Bars *above* the line are excluded; bars *below* are STEP-detectable. The vestigial-phase bar (η=1) is far above; the relics bar (η~10⁻¹⁸) is at the STEP target.

In [5]:
# viz-ref: fig_ep_violation_matrix
fig_ep_violation_matrix().show()

## 6. Honest scope and what this paper does not claim

**What this work proves.** Given the six Phase-5x DM mechanisms as currently formulated in the project, EP violation is a *vestigial-only* phenomenon. The classification is encoded as 24 machine-checked Lean theorems (12 original + 12 strengthening pass) with zero `sorry` statements.

**What this work does not prove.** That the project's six mechanisms exhaust the dark-sector possibilities. There may be other DM models — outside this project's scope — that violate EP via different physics. The conclusion is *substrate-specific*: "In this substrate, EP violation is vestigial-only." It is not a universal claim about all DM models.

**What would falsify the bridge.**

1. STEP detecting $\eta$ in the $10^{-15}$–$10^{-18}$ window with a *species-dependent* signature inconsistent with vestigial relics — would identify a different mechanism not yet in the project.
2. STEP returning a null result at $\eta < 10^{-19}$ — would exclude the vestigial-relics mechanism *within the project's substrate*, leaving the four EP-respecting candidates as the live set.
3. A redefinition of vestigial gravity that makes the $\eta \sim 10^{-18}$ prediction wrong — would update the classification.

**Why this matters strategically.** EP precision experiments have been moving toward $\eta \sim 10^{-18}$ for over a decade. Without this paper, that effort had no specific theoretical target in our framework. With it, the projected STEP-class design sensitivity *is* the relics signature, with a falsifiable null prediction backing it up.

## 7. Where to go next

- **Paper:** `papers/paper34_equivalence_principle/paper_draft.tex` — full classification with abstract hedge on STEP projection.
- **Lean module:** `lean/SKEFTHawking/EquivalencePrinciple.lean` — 24 theorems (12 original + 12 strengthening).
- **Companion technical notebook:** `Phase6c3_EquivalencePrinciple_Technical.ipynb` — full Lean theorem inventory and `violatesAt_mono` monotonicity walkthrough.
- **Cross-bridges:** `FangGuTorsionDM.fg_cdm_obstruction` (kinematic failure of FangGu, not EP); `ClassificationTableDark.MicroscopeStatus` (status table for all six).